In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_IGI_Airport_(T3)_Delhi_IMD_2023.xlsx")

In [3]:
df.head()
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    31 non-null     float64
 2   February   31 non-null     float64
 3   March      35 non-null     float64
 4   April      32 non-null     float64
 5   May        32 non-null     float64
 6   June       27 non-null     float64
 7   July       32 non-null     float64
 8   August     30 non-null     float64
 9   September  28 non-null     float64
 10  October    31 non-null     float64
 11  November   33 non-null     float64
 12  December   34 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


(41, 13)

In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))


In [6]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())


In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,145.000000,111.0,137.000000,138.0,76.0,88.0,97.125,101.8,93.357143,144.0,357.0,375.0
1,2,315.000000,133.0,120.885714,158.0,58.0,80.0,97.125,101.8,127.000000,136.0,383.0,362.0
2,3,214.903226,132.0,112.000000,181.0,110.0,98.0,102.000,101.8,132.000000,167.0,492.0,286.0
3,4,335.000000,171.0,98.000000,100.0,72.0,121.0,115.000,108.0,126.000000,164.0,424.0,309.0
4,5,301.000000,169.0,102.000000,111.0,150.0,114.0,115.000,108.0,119.000000,171.0,468.0,305.0


In [8]:

df_ml_ready.info()

df_ml_ready.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        31 non-null     int64  
 1   January    31 non-null     float64
 2   February   31 non-null     float64
 3   March      31 non-null     float64
 4   April      31 non-null     float64
 5   May        31 non-null     float64
 6   June       31 non-null     float64
 7   July       31 non-null     float64
 8   August     31 non-null     float64
 9   September  31 non-null     float64
 10  October    31 non-null     float64
 11  November   31 non-null     float64
 12  December   31 non-null     float64
dtypes: float64(12), int64(1)
memory usage: 3.3 KB


(31, 13)